# Stanford Congressional Record Dataset (2001-2017)

This notebook downloads, processes, and validates the Stanford
(Gentzkow, Shapiro & Taddy) Congressional Record dataset, covering the
107th through 114th sessions of Congress. Each speech in the resulting
table is linked to a speaker, party, ICPSR identifier (where resolved),
chamber, and date.

## Structure of this notebook

1. Connect to Google Drive
2. Download and unzip the raw Stanford data
3. Load and combine all sessions into one table
4. Check the actual date range covered by the data
5. Speaker-to-ICPSR resolution
6. Save the result

## Output location

This notebook writes its output to a working folder in Google Drive
used for in-progress data (see the Drive connection cell below). That
folder is not the one shared with collaborators. Promoting a finished,
validated file to the shared folder is a manual step, done outside
this notebook, so that collaborators are never affected by a file
being rewritten mid-run.

## Section 1: Connect to Google Drive

Two Drive folders are used across this project:

- **Working folder** ("... (raw)"): where this notebook automatically
  saves its output. Intended for in-progress work.
- **Shared folder** ("... (shared)"): a separate location that only
  ever receives a manually copied file, once a version of the dataset
  is confirmed ready for use by collaborators. No notebook writes to
  this folder automatically, so collaborators' in-progress work is
  never overwritten by a background process.

Only the working folder is used for saving in this notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

WORKING_FOLDER = "/content/drive/MyDrive/Work Project 2026: Datasets (raw)"
SHARED_FOLDER = "/content/drive/MyDrive/Work Project 2026: Datasets (shared)"

import os
os.makedirs(WORKING_FOLDER, exist_ok=True)
os.makedirs(SHARED_FOLDER, exist_ok=True)  # created if missing, never written to automatically

print(f"Working folder: {WORKING_FOLDER}")
print(f"Shared folder (manual copies only): {SHARED_FOLDER}")

Mounted at /content/drive
Working folder: /content/drive/MyDrive/Work Project 2026: Datasets (raw)
Shared folder (manual copies only): /content/drive/MyDrive/Work Project 2026: Datasets (shared)


In [2]:
# Quick sanity check: write a small test file to the working folder and
# confirm in the browser (drive.google.com) that it actually appears
# there, before relying on this connection for a long-running task. A
# Drive mount can appear connected in this session while writes are
# not actually reaching the remote file (observed previously with this
# same setup), so this check is done before any real data is written.
test_path = f"{WORKING_FOLDER}/verify_mount_test.txt"
with open(test_path, "w") as f:
    f.write("test")
print("File written. Check drive.google.com for 'verify_mount_test.txt' before continuing.")

File written. Check drive.google.com for 'verify_mount_test.txt' before continuing.


## Section 2: Download and unzip the raw dataset

The dataset is distributed by Stanford as a single compressed archive
covering all available sessions of the Congressional Record's daily
edition. Sessions of interest here are 107 through 114 (corresponding
to the years 2001 onward).

A "session" corresponds to one two-year Congress: the 107th Congress
covers 2001-2002, the 108th covers 2003-2004, and so on, through the
114th.

In [3]:
# The archive is a few gigabytes; downloading and unzipping can take
# several minutes. "-o" overwrites without prompting if this cell is
# run again; "-q" suppresses the file-by-file listing during extraction.
!wget https://stacks.stanford.edu/file/druid:md374tz9962/hein-daily.zip
!unzip -o -q hein-daily.zip -d hein-daily

--2026-09-29 18:17:15--  https://stacks.stanford.edu/file/druid:md374tz9962/hein-daily.zip
Resolving stacks.stanford.edu (stacks.stanford.edu)... 171.67.37.91
Connecting to stacks.stanford.edu (stacks.stanford.edu)|171.67.37.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘hein-daily.zip’

hein-daily.zip          [            <=>     ]   2.62G  13.5MB/s    in 3m 25s  

2026-09-29 18:20:41 (13.1 MB/s) - ‘hein-daily.zip’ saved [2817259858]



## Section 3: Load and combine all sessions

Each session is stored as three separate pipe-delimited files: the raw
speech text, speech-level metadata (including date), and a speaker map
linking each speech to a specific member of Congress and their party.
These three files are merged on their shared `speech_id`, and speeches
with no matched speaker (typically procedural entries such as a clerk
reading a bill title) are dropped.

Text encoding is `latin-1`, not UTF-8, since the source text was
digitized from printed volumes via OCR.

In [4]:
import pandas as pd


def load_and_clean_session(session_num, inner_path="hein-daily/hein-daily"):
    """
    Loads and cleans one two-year Congress session of speeches.

    Reads the three per-session files (speeches, description/metadata,
    speaker map), merges them on speech_id, drops speeches with no
    matched speaker, and returns the combined table.

    The SpeakerMap file's own 'firstname' field is pulled in as
    'speaker_first_name' - the pre-existing 'first_name' column
    (from the speeches/description files) was found to be an unpopulated
    placeholder ("Unknown" for every row checked), not a usable value.
    'speaker_first_name' is the reliable field for any first-name-based
    matching (see Section 5).
    """
    session_str = str(session_num).zfill(3)

    speeches = pd.read_csv(
        f"{inner_path}/speeches_{session_str}.txt", sep="|", dtype=str,
        on_bad_lines="skip", quoting=3, encoding="latin-1",
    )
    descr = pd.read_csv(
        f"{inner_path}/descr_{session_str}.txt", sep="|", dtype=str,
        on_bad_lines="skip", quoting=3, encoding="latin-1",
    )
    speakermap = pd.read_csv(
        f"{inner_path}/{session_str}_SpeakerMap.txt", sep="|", dtype=str,
        on_bad_lines="skip", quoting=3, encoding="latin-1",
    )

    merged = speeches.merge(descr, on="speech_id", how="left")
    merged = merged.merge(
        speakermap[["speech_id", "speakerid", "party", "state", "chamber", "firstname"]]
        .rename(columns={"firstname": "speaker_first_name"}),
        on="speech_id", how="left", suffixes=("", "_map"),
    )

    clean = merged[merged["speakerid"].notna()].copy()
    clean["congress"] = session_num
    return clean


def load_all_sessions(start_session, end_session, inner_path="hein-daily/hein-daily"):
    """Loads and combines every session from start_session to end_session (inclusive)."""
    frames = []
    for session in range(start_session, end_session + 1):
        df = load_and_clean_session(session, inner_path)
        print(f"Session {session}: {df.shape[0]} speeches loaded")
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [5]:
historical_dataset = load_all_sessions(107, 114)
print(f"\nTotal speeches loaded: {historical_dataset.shape[0]}")

Session 107: 116866 speeches loaded
Session 108: 127963 speeches loaded
Session 109: 117130 speeches loaded
Session 110: 130090 speeches loaded
Session 111: 94641 speeches loaded
Session 112: 91060 speeches loaded
Session 113: 77621 speeches loaded
Session 114: 67970 speeches loaded

Total speeches loaded: 823341


## Section 4: Check the actual date range

Stanford's own documentation describes the daily edition of the
Congressional Record as covering sessions 97 through 114 (1981-2017).
The actual end date present in a specific downloaded file can differ
from a source's general documented range, so the real minimum and
maximum dates are checked directly against the loaded data, broken
down by session, rather than assumed from the documentation alone.

This matters for combining this dataset with the govinfo-based dataset
(see the govinfo notebook), which starts on January 1, 2017: any gap
between where this dataset actually ends and where the other begins
needs to be identified and accounted for.

In [6]:
historical_dataset["date_parsed"] = pd.to_datetime(
    historical_dataset["date"], format="%Y%m%d", errors="coerce"
)

print(f"Earliest speech: {historical_dataset['date_parsed'].min()}")
print(f"Latest speech:   {historical_dataset['date_parsed'].max()}")

print("\nLatest date per Congress session:")
print(historical_dataset.groupby("congress")["date_parsed"].max())

Earliest speech: 2001-01-03 00:00:00
Latest speech:   2016-09-09 00:00:00

Latest date per Congress session:
congress
107   2002-11-22
108   2004-12-08
109   2006-12-08
110   2008-12-12
111   2010-12-22
112   2013-01-03
113   2014-12-16
114   2016-09-09
Name: date_parsed, dtype: datetime64[ns]


In [7]:
# Checks the RAW session 114 file directly, independent of the merge
# and cleaning logic above. This distinguishes two possibilities if a
# gap is found: the raw file itself stops early (a characteristic of
# the released Stanford data), versus the merge/cleaning step above
# dropping rows that are actually present in the raw file.
descr_114 = pd.read_csv(
    "hein-daily/hein-daily/descr_114.txt", sep="|", dtype=str,
    on_bad_lines="skip", quoting=3, encoding="latin-1",
)
descr_114["date_parsed"] = pd.to_datetime(descr_114["date"], format="%Y%m%d", errors="coerce")

print(f"Raw session 114 file - earliest date: {descr_114['date_parsed'].min()}")
print(f"Raw session 114 file - latest date:   {descr_114['date_parsed'].max()}")
print(f"Total rows in raw file: {len(descr_114)}")

Raw session 114 file - earliest date: 2015-01-06 00:00:00
Raw session 114 file - latest date:   2016-09-09 00:00:00
Total rows in raw file: 112290


## Section 5: Speaker-to-ICPSR resolution

Unlike the govinfo dataset, this dataset already provides each
speaker's first and last name as separate columns, so no name-parsing
is needed here - only a lookup against a reference source. The same
`congress-legislators` project used for the govinfo dataset
(github.com/unitedstates/congress-legislators) is used here, matched
on first name, last name, and date (checking which term, if any,
covers the speech's date), with state used as an additional
tie-breaker when a first+last name match is ambiguous.

A speaker is only assigned an ICPSR id when exactly one legislator
matches; if more than one sitting member shares the same first and
last name (rare) and the state does not resolve the ambiguity, no id
is assigned rather than guessing.

As a byproduct of this lookup, the matched legislator's party for that
term is also available. This is recorded in a separate
`party_crosscheck` column, distinct from Stanford's own `party`
column, so the two independently-derived party assignments can be
compared as a data-quality check without altering Stanford's original
value.

In [8]:
# Reference data: same source used for the govinfo dataset.
!wget -q https://raw.githubusercontent.com/unitedstates/congress-legislators/main/legislators-current.yaml
!wget -q https://raw.githubusercontent.com/unitedstates/congress-legislators/main/legislators-historical.yaml

import yaml
import unicodedata
from collections import defaultdict

with open("legislators-current.yaml") as f:
    legislators_current = yaml.safe_load(f)
with open("legislators-historical.yaml") as f:
    legislators_historical = yaml.safe_load(f)

all_legislators = legislators_current + legislators_historical
print("Total legislators loaded (current + historical):", len(all_legislators))

Total legislators loaded (current + historical): 12770


In [9]:
def normalize_name(s):
    """Lowercases and strips accents, so e.g. 'Barragan' and 'Barragan' with an accent compare equal."""
    if s is None:
        return ""
    s = str(s).lower()
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c))


# Stanford stores state as a full name (e.g. "Illinois"); the
# congress-legislators reference data stores state as a two-letter
# abbreviation (e.g. "IL") - both are normalized to the same lowercase
# abbreviation before comparing, since a naive text comparison between
# the two formats would never match.
STATE_NAME_TO_ABBR = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia",
    "kansas": "ks", "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms", "missouri": "mo",
    "montana": "mt", "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj",
    "new mexico": "nm", "new york": "ny", "north carolina": "nc", "north dakota": "nd", "ohio": "oh",
    "oklahoma": "ok", "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt",
    "virginia": "va", "washington": "wa", "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "district of columbia": "dc", "puerto rico": "pr", "guam": "gu", "virgin islands": "vi",
    "american samoa": "as", "northern mariana islands": "mp",
}


def normalize_state(s):
    """Normalizes a state value to a lowercase two-letter abbreviation, whether given as a full name or already an abbreviation."""
    n = normalize_name(s)
    return STATE_NAME_TO_ABBR.get(n, n)


# Index legislators by normalized last name, so each row only has to
# check a small number of candidates instead of scanning all ~12,700
# legislators. Term start/end dates and state are pre-parsed and
# pre-normalized ONCE here, rather than inside the per-row matching
# function - repeatedly re-parsing the same, unchanging term dates for
# every one of 800,000+ rows was found to make the matching step take
# multiple hours; pre-computing them once brings this down to
# something on the order of a couple of minutes.
legislators_by_last = defaultdict(list)
for leg in all_legislators:
    parsed_terms = [
        {
            "start": pd.to_datetime(term["start"]),
            "end": pd.to_datetime(term["end"]),
            "party": term.get("party"),
            "state": normalize_state(term.get("state", "")),
        }
        for term in leg["terms"]
    ]
    legislators_by_last[normalize_name(leg["name"]["last"])].append({
        "first_name_norm": normalize_name(leg["name"]["first"]),
        "icpsr": leg.get("id", {}).get("icpsr"),
        "terms": parsed_terms,
    })


def resolve_icpsr_and_party(last_name, first_name, state, date):
    """
    Returns (icpsr_id, crosscheck_party) for one speech, or (None, None)
    if no single legislator can be matched unambiguously.

    Matches by last name and date (checking which pre-parsed term, if
    any, covers the speech date) first; if more than one legislator
    with that surname served at that time, first name and then state
    are used to narrow down to a single match. Both first_name and
    state can be "Unknown" (Stanford leaves this value when the source
    record itself did not specify it), in which case that narrowing
    step is skipped rather than treated as a real value to match against.
    """
    if date is None or last_name is None:
        return None, None

    candidates = legislators_by_last.get(normalize_name(last_name), [])
    if not candidates:
        return None, None

    def term_at_date(leg):
        for term in leg["terms"]:
            if term["start"] <= date <= term["end"]:
                return term
        return None

    results = [(leg, term_at_date(leg)) for leg in candidates]
    results = [(leg, term) for leg, term in results if term is not None]

    if len(results) > 1 and first_name and normalize_name(first_name) != "unknown":
        first_norm = normalize_name(first_name)
        narrowed = [(leg, term) for leg, term in results if leg["first_name_norm"] == first_norm]
        if narrowed:
            results = narrowed

    if len(results) > 1 and state and normalize_name(state) != "unknown":
        state_norm = normalize_state(state)
        narrowed = [(leg, term) for leg, term in results if term["state"] == state_norm]
        if narrowed:
            results = narrowed

    if len(results) != 1:
        return None, None

    leg, term = results[0]
    return leg["icpsr"], term["party"]

In [10]:
# Applied row by row, with a progress bar (tqdm) so progress is
# visible rather than a silent wait. With the pre-parsed index above,
# this should complete in a few minutes rather than several hours.
from tqdm.auto import tqdm
tqdm.pandas()

resolved = historical_dataset.progress_apply(
    lambda row: resolve_icpsr_and_party(row["last_name"], row["speaker_first_name"], row["state"], row["date_parsed"]),
    axis=1,
)
historical_dataset["icpsr"] = resolved.apply(lambda pair: pair[0])
historical_dataset["party_crosscheck"] = resolved.apply(lambda pair: pair[1])

icpsr_coverage = historical_dataset["icpsr"].notna().mean()
print(f"ICPSR resolved for {icpsr_coverage:.1%} of speeches")

  0%|          | 0/823341 [00:00<?, ?it/s]

ICPSR resolved for 97.7% of speeches


In [11]:
# Data-quality check: how often does the independently-derived
# party_crosscheck column agree with Stanford's own party column,
# among speeches where both are available? Stanford uses single-letter
# codes (D/R/I/...) while party_crosscheck uses full words, so the
# values are mapped to a common form before comparing.
party_code_map = {"Democrat": "D", "Republican": "R", "Independent": "I"}

both_present = historical_dataset[historical_dataset["party_crosscheck"].notna()].copy()
both_present["party_crosscheck_mapped"] = both_present["party_crosscheck"].map(party_code_map)

agreement = (both_present["party"] == both_present["party_crosscheck_mapped"]).mean()
print(f"Rows with both party values available: {len(both_present)}")
print(f"Agreement between Stanford's party and the ICPSR crosscheck: {agreement:.1%}")

mismatches = both_present[both_present["party"] != both_present["party_crosscheck_mapped"]]
print(f"\nSample of disagreements ({len(mismatches)} total):")
print(mismatches[["date", "speaker", "first_name", "party", "party_crosscheck"]].head(10))

Rows with both party values available: 810632
Agreement between Stanford's party and the ICPSR crosscheck: 100.0%

Sample of disagreements (90 total):
          date           speaker first_name party party_crosscheck
2520  20010213      Mr. CRENSHAW    Unknown     I       Republican
3187  20010227      Mr. CRENSHAW    Unknown     I       Republican
3364  20010228      Mr. CRENSHAW    Unknown     I       Republican
3920  20010306  Mr. ACEVEDO-VILA    Unknown     A               AL
4500  20010307      Mr. CRENSHAW    Unknown     I       Republican
5516  20010313  Mr. ACEVEDO-VILA    Unknown     A               AL
8039  20010322      Mr. CRENSHAW    Unknown     I       Republican
9165  20010327      Mr. CRENSHAW    Unknown     I       Republican
9313  20010327      Mr. CRENSHAW    Unknown     I       Republican
9500  20010328      Mr. CRENSHAW    Unknown     I       Republican


## Section 6: Save the result

The combined table is saved as a parquet file (a compressed,
efficient format for large data tables) in the working Drive folder.

Before overwriting the working folder's file, the row count and
session coverage are checked to avoid replacing a complete file with
an incomplete one from an interrupted run.

In [12]:
# Safety check: only overwrite the saved file if the data looks
# complete, so an interrupted or partial run cannot silently replace a
# good existing file with a worse one.
expected_min_speeches = 500_000
expected_sessions = set(range(107, 115))
actual_sessions = set(historical_dataset["congress"].unique())

if historical_dataset.shape[0] < expected_min_speeches:
    print(f"WARNING: only {historical_dataset.shape[0]} speeches found - "
          f"expected at least {expected_min_speeches}. NOT saving.")
elif actual_sessions != expected_sessions:
    missing = expected_sessions - actual_sessions
    print(f"WARNING: missing session(s) {missing} - NOT saving.")
else:
    output_path = f"{WORKING_FOLDER}/congress_speeches_2001_2017.parquet"
    historical_dataset.to_parquet(output_path)
    print(f"Looks complete ({historical_dataset.shape[0]} speeches, "
          f"sessions {sorted(actual_sessions)}). Saved to {output_path}")

Looks complete (823341 speeches, sessions [np.int64(107), np.int64(108), np.int64(109), np.int64(110), np.int64(111), np.int64(112), np.int64(113), np.int64(114)]). Saved to /content/drive/MyDrive/Work Project 2026: Datasets (raw)/congress_speeches_2001_2017.parquet


## Notes

- Columns in the saved file: `speech_id`, `speech`, `chamber`, `date`
  (YYYYMMDD), `speaker`, `first_name`, `state`, `gender`, `word_count`,
  `speakerid`, `party` (D/R/I/P/A), `congress`, `icpsr`,
  `party_crosscheck`.
- `speakerid` is a Stanford-internal identifier (session + a 5-digit
  person ID + chamber flag), not a standard cross-dataset identifier.
  `icpsr` (added in Section 5) fills this role, resolved against the
  same reference source used for the govinfo dataset.
- `icpsr` and `party_crosscheck` are `null` where no legislator could
  be matched unambiguously (see Section 5). As with the govinfo
  dataset, ICPSR ids are only ever assigned to members who cast
  recorded floor votes; non-voting territorial delegates structurally
  never receive one, regardless of match confidence otherwise.
- `party_crosscheck` is an independently-derived party value and is
  distinct from Stanford's own `party` column; Section 5 checks how
  often the two agree, as a data-quality signal, without altering
  Stanford's original `party` value.
- Promoting this file to the shared Drive folder, for use by
  collaborators, is a manual step performed outside this notebook.